# SelvaSonic — Entrenamiento Attention V5 (Fase 3: Class Weights + Label Smoothing)

## Objetivo

Entrenar **SelvaSonicCNNAttention v2** con dos mejoras sobre el v1:

1. **Class Weights** — Pesos inversamente proporcionales a la frecuencia de cada clase.
   Mitiga el desbalance severo (Chordeiles: 240 clips vs no_ave: 2,240 clips).
   Precomputados con `scripts/calcular_class_weights.py`.

2. **Label Smoothing α=0.1** — Reduce el overconfidence bias detectado en el notebook 11
   (el v1 predecía con probabilidad >0.95 en audios ambiguos del Amazonas).

## Cambios respecto al V4 (attention v1)

| Item | V4 (attention v1) | V5 (attention v2) |
|------|-------------------|-------------------|
| Class weights | No | Sí (`USE_CLASS_WEIGHTS=True`) |
| Label smoothing | 0.1 (igual) | 0.1 |
| Checkpoint dir | `checkpoints_activos_attention/` | `checkpoints_activos_attention_v2/` |
| Loss | `nn.CrossEntropyLoss(ls=0.1)` | `build_loss()` de `src/loss.py` |

Ver `docs/decisiones_metodologicas.md` para la justificación completa.

## Hipótesis

Con class weights esperamos que las clases minoritarias (Chordeiles, Rupornis) mejoren
su recall. Con label smoothing el modelo debería estar mejor calibrado en audios ambiguos.

## Celda 1 — Montar Drive y verificar GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("NO HAY GPU. Cambia el runtime a T4 GPU")

## Celda 2 — Descomprimir código y datos

In [ ]:
import os
import zipfile
import shutil

PROYECTO = '/content/SelvaSonic-ML'
DRIVE_BASE = '/content/drive/MyDrive/SelvaSonic_Proyecto'

# Siempre re-descomprimir en V5 para garantizar que src/loss.py está incluido
FORZAR_REDESCOMPRESION = True

if FORZAR_REDESCOMPRESION and os.path.exists(PROYECTO):
    print(f"Borrando {PROYECTO} para re-descomprimir version con loss.py...")
    shutil.rmtree(PROYECTO)

if os.path.exists(os.path.join(PROYECTO, 'src', 'loss.py')) and \
   os.path.exists(os.path.join(PROYECTO, 'data', 'raw')):
    print("OK Proyecto ya restaurado, saltando descompresion")
else:
    os.makedirs(PROYECTO, exist_ok=True)

    print("[1/3] Descomprimiendo codigo (version con src/loss.py)...")
    with zipfile.ZipFile(f'{DRIVE_BASE}/codigo.zip', 'r') as z:
        z.extractall(PROYECTO)

    print("[2/3] Descomprimiendo datos (puede tardar 2-3 min)...")
    os.makedirs(f'{PROYECTO}/data/raw', exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/data_raw.zip', 'r') as z:
        z.extractall(f'{PROYECTO}/data/raw')

    print("[3/3] Fix de backslash en nombres (bug de zip de Windows)...")
    for base in [PROYECTO, f'{PROYECTO}/data/raw']:
        archivos_con_bs = [i for i in os.listdir(base) if '\\' in i]
        for nombre_viejo in archivos_con_bs:
            nombre_nuevo = nombre_viejo.replace('\\', '/')
            ruta_vieja = os.path.join(base, nombre_viejo)
            ruta_nueva = os.path.join(base, nombre_nuevo)
            os.makedirs(os.path.dirname(ruta_nueva), exist_ok=True)
            if os.path.isdir(ruta_vieja):
                if os.path.exists(ruta_nueva):
                    shutil.rmtree(ruta_nueva)
                shutil.move(ruta_vieja, ruta_nueva)
            else:
                shutil.move(ruta_vieja, ruta_nueva)

    print("OK Proyecto y datos listos")

os.chdir(PROYECTO)
print(f"\nDirectorio actual: {os.getcwd()}")

# Verificar que los modulos clave existen
for modulo in ['src/model.py', 'src/loss.py', 'src/class_weights.py']:
    if os.path.exists(os.path.join(PROYECTO, modulo)):
        print(f"OK {modulo} detectado")
    else:
        raise RuntimeError(f"ERROR: {modulo} NO esta en el codigo. Re-sube codigo.zip a Drive.")

## Celda 3 — Instalar dependencias

In [ ]:
!pip install -q librosa==0.10.1 soundfile pyyaml tensorboard
print("OK dependencias instaladas")

## Celda 4 — Importar módulos

In [ ]:
import sys
sys.path.insert(0, '/content/SelvaSonic-ML')

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNNAttention
from src.logger import TrainingLogger
from src.loss import build_loss          # Fase 3: construccion de loss balanceada
from src import config

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import json
import time
from datetime import datetime

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("OK Modulos importados (incluyendo build_loss de src/loss.py)")

## Celda 5 — Configuración del entrenamiento

Las flags de la Fase 3 se cargan desde `src/config.py` para garantizar
consistencia. El resto de hiperparámetros es idéntico al v1.

In [ ]:
# === FLAGS FASE 3 (desde config.py) ===
USE_CLASS_WEIGHTS = config.USE_CLASS_WEIGHTS       # True
LABEL_SMOOTHING   = config.LABEL_SMOOTHING_ALPHA   # 0.1

# === Hiperparametros de entrenamiento (identicos al v1 para comparacion justa) ===
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_T_MAX = EPOCHS
NUM_WORKERS = 2
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

LOG_CM_EVERY = config.LOG_CONFUSION_MATRIX_EVERY_N_EPOCHS  # 5

# Run name con sufijo v2
RUN_NAME = f"attention_S4_v2_{datetime.now().strftime('%Y%m%d_%H%M')}"
DRIVE_RUN_DIR = f'{DRIVE_BASE}/runs/{RUN_NAME}'
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

# NUEVA carpeta de checkpoints — distinta del v1, NO lo sobrescribe
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints_activos_attention_v2'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
LATEST_CKPT  = f'{DRIVE_CKPT_DIR}/latest.pth'
BEST_CKPT    = f'{DRIVE_CKPT_DIR}/best.pth'
HISTORY_JSON = f'{DRIVE_CKPT_DIR}/history.json'

TB_LOG_DIR = f'{DRIVE_RUN_DIR}/tensorboard'

print(f"Run name:    {RUN_NAME}")
print(f"Run dir:     {DRIVE_RUN_DIR}")
print(f"Checkpoints: {DRIVE_CKPT_DIR}")
print()
print("=== FLAGS FASE 3 ===")
print(f"  USE_CLASS_WEIGHTS: {USE_CLASS_WEIGHTS}")
print(f"  LABEL_SMOOTHING:   {LABEL_SMOOTHING}")
print()
print("Hiperparametros (identicos al v1):")
print(f"  EPOCHS:    {EPOCHS} | BATCH: {BATCH_SIZE} | LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY}")
print(f"  PATIENCE:  {EARLY_STOPPING_PATIENCE} | LOG_CM_EVERY: {LOG_CM_EVERY}")

## Celda 5b — Verificar y copiar `class_weights.pt`

El tensor de class weights debe existir en Drive antes de continuar.
Si no existe, esta celda imprime las instrucciones exactas para subirlo.

In [ ]:
# Ruta en Drive donde el usuario debe subir el archivo
DRIVE_WEIGHTS_PATH = f'{DRIVE_BASE}/data/class_weights.pt'
LOCAL_WEIGHTS_PATH = f'{PROYECTO}/data/class_weights.pt'

if not os.path.exists(DRIVE_WEIGHTS_PATH):
    print("=" * 70)
    print("  ERROR: class_weights.pt NO encontrado en Drive.")
    print("=" * 70)
    print()
    print("Para generar y subir el archivo:")
    print()
    print("  1. En tu PC local, ejecuta:")
    print("       python scripts/calcular_class_weights.py")
    print()
    print("  2. Sube el archivo generado a Google Drive en la ruta:")
    print(f"       {DRIVE_WEIGHTS_PATH}")
    print("     (crea la carpeta 'data/' dentro de SelvaSonic_Proyecto si no existe)")
    print()
    print("  3. Vuelve a ejecutar esta celda.")
    raise FileNotFoundError(f"No encontrado: {DRIVE_WEIGHTS_PATH}")

# Copiar al proyecto local para que config.CLASS_WEIGHTS_PATH apunte correctamente
os.makedirs(f'{PROYECTO}/data', exist_ok=True)
shutil.copy2(DRIVE_WEIGHTS_PATH, LOCAL_WEIGHTS_PATH)
size_kb = os.path.getsize(LOCAL_WEIGHTS_PATH) / 1024

# Verificar el contenido del tensor
import torch as _torch
_payload = _torch.load(LOCAL_WEIGHTS_PATH, weights_only=False, map_location='cpu')
_weights = _payload['weights']
print("OK class_weights.pt copiado y verificado")
print(f"   Origen:      {DRIVE_WEIGHTS_PATH}")
print(f"   Destino:     {LOCAL_WEIGHTS_PATH}  ({size_kb:.1f} KB)")
print(f"   Num clases:  {_payload.get('num_classes', '?')}")
print(f"   Total clips: {_payload.get('total_samples', '?')}")
print(f"   Pesos min/max: {_weights.min():.4f} / {_weights.max():.4f}")
print(f"   Pesos: {[round(float(w), 4) for w in _weights]}")

## Celda 6 — DataLoaders

In [ ]:
RAW_DATA_DIR = f'{PROYECTO}/data/raw'

train_loader, val_loader, test_loader, label_map = create_dataloaders(
    raw_data_dir=RAW_DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_state=SEED,    # MISMA semilla que v1 -> mismo split
    verbose=True,
)

NUM_CLASSES = len(label_map)
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

print(f"\nTrain: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Clases ({NUM_CLASSES}): {class_names}")

x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch shape: {tuple(x_sample.shape)}")

## Celda 7 — Modelo, loss balanceada, optimizer y scheduler

La loss se construye con `build_loss()` de `src/loss.py`. Si `USE_CLASS_WEIGHTS=True`,
carga el tensor precomputado y lo pasa como argumento `weight` a CrossEntropyLoss.

In [ ]:
model = SelvaSonicCNNAttention(num_classes=NUM_CLASSES).to(device)
n_params = model.count_parameters()
print(f"Modelo: SelvaSonicCNNAttention")
print(f"Parametros entrenables: {n_params:,}")

# Smoke test forward
model.eval()
with torch.no_grad():
    out_test = model(x_sample.to(device))
assert out_test.shape == (BATCH_SIZE, NUM_CLASSES), f"Shape incorrecto: {out_test.shape}"
print(f"OK Forward: {tuple(x_sample.shape)} -> {tuple(out_test.shape)}")

# Loss con class weights + label smoothing (via src/loss.py)
criterion, loss_info = build_loss(
    use_class_weights=USE_CLASS_WEIGHTS,
    label_smoothing=LABEL_SMOOTHING,
    class_weights_path=config.CLASS_WEIGHTS_PATH,
    num_classes=NUM_CLASSES,
    device=device,
)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX, eta_min=1e-6)

print(f"\nLoss configurada:")
print(f"  use_class_weights: {loss_info['use_class_weights']}")
print(f"  label_smoothing:   {loss_info['label_smoothing']}")
print(f"  weights_min:       {loss_info['weights_min']:.4f}" if loss_info['weights_min'] else "  weights_min: N/A")
print(f"  weights_max:       {loss_info['weights_max']:.4f}" if loss_info['weights_max'] else "  weights_max: N/A")
print(f"\nOptimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR(T_max={SCHEDULER_T_MAX})")

## Celda 8 — Inicializar TrainingLogger (TensorBoard) y loguear config de loss

In [ ]:
hparams_dict = {
    'model': 'SelvaSonicCNNAttention_v2',
    'num_heads': config.ATTENTION_NUM_HEADS,
    'attention_dropout': config.ATTENTION_DROPOUT,
    'classifier_dropout': config.DROPOUT,
    'batch_size': BATCH_SIZE,
    'lr': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'use_class_weights': USE_CLASS_WEIGHTS,
    'weights_min': str(round(loss_info['weights_min'], 4)) if loss_info['weights_min'] else 'N/A',
    'weights_max': str(round(loss_info['weights_max'], 4)) if loss_info['weights_max'] else 'N/A',
    'epochs_max': EPOCHS,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'n_params': n_params,
}

tb_logger = TrainingLogger(
    log_dir=TB_LOG_DIR,
    run_name=RUN_NAME,
    label_map=label_map,
    hparams=hparams_dict,
    enabled=True,
)

# Loguear configuracion de loss en TensorBoard (step 0)
# Permite comparar graficamente entre v1 y v2 en la pestana SCALARS
if tb_logger._writer is not None:
    tb_logger._writer.add_scalar("loss_config/use_class_weights", float(USE_CLASS_WEIGHTS), 0)
    tb_logger._writer.add_scalar("loss_config/label_smoothing", LABEL_SMOOTHING, 0)
    if loss_info['weights_min'] is not None:
        tb_logger._writer.add_scalar("loss_config/weights_min", loss_info['weights_min'], 0)
        tb_logger._writer.add_scalar("loss_config/weights_max", loss_info['weights_max'], 0)

try:
    tb_logger.log_model_graph(model, x_sample.to(device))
    print("OK Grafo del modelo logueado")
except Exception as e:
    print(f"AVISO: no se pudo loguear el grafo ({e}). No es critico.")

print(f"OK TensorBoard inicializado en: {TB_LOG_DIR}")
print(f"OK loss_config logueado (use_class_weights={USE_CLASS_WEIGHTS}, ls={LABEL_SMOOTHING})")

## Celda 9 — Funciones auxiliares de checkpoint, entrenamiento y evaluación

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, history, best_val_acc, label_map):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_val_acc': best_val_acc,
        'label_map': label_map,
        'hparams': hparams_dict,
        'loss_info': loss_info,       # guardamos config de loss para trazabilidad
    }, path)


def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'] + 1, ckpt['history'], ckpt['best_val_acc']


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_y, all_pred = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
        if return_preds:
            all_y.append(y.cpu())
            all_pred.append(pred.cpu())
    if return_preds:
        return total_loss / total, correct / total, torch.cat(all_y).numpy(), torch.cat(all_pred).numpy()
    return total_loss / total, correct / total


print("OK Funciones definidas")

## Celda 10 — Resume automático

In [ ]:
start_epoch = 0
best_val_acc = 0.0
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'lr': [], 'epoch_time_s': [],
}

RESUME = True

if RESUME and os.path.exists(LATEST_CKPT):
    print(f"Checkpoint encontrado en {LATEST_CKPT}")
    start_epoch, history, best_val_acc = load_checkpoint(
        LATEST_CKPT, model, optimizer, scheduler
    )
    print(f"OK Reanudando desde epoca {start_epoch} (best_val_acc previo: {best_val_acc:.4f})")
else:
    if not RESUME:
        for f in [LATEST_CKPT, BEST_CKPT, HISTORY_JSON]:
            if os.path.exists(f):
                os.remove(f)
        print("RESUME=False, empezando de cero")
    else:
        print("No hay checkpoint previo, empezando de cero (epoca 0)")

## Celda 11 — Bucle de entrenamiento

Misma estructura robusta del V4 (checkpoint a Drive cada época, resume automático,
TensorBoard), pero con la loss balanceada y el bug de `log_confusion_matrix` corregido
(ahora pasa `y_true`, `y_pred`, `class_names` como keyword args).

In [ ]:
print("=" * 75)
print(f"ENTRENAMIENTO ATTENTION V2 — {RUN_NAME}")
print(f"Desde epoca {start_epoch} hasta {EPOCHS}")
print(f"USE_CLASS_WEIGHTS={USE_CLASS_WEIGHTS} | LABEL_SMOOTHING={LABEL_SMOOTHING}")
print("=" * 75)

epochs_without_improvement = 0

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # En epocas de confusion matrix, pedimos predicciones tambien
    if (epoch + 1) % LOG_CM_EVERY == 0:
        val_loss, val_acc, y_val_true, y_val_pred = evaluate(
            model, val_loader, criterion, device, return_preds=True
        )
    else:
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        y_val_true, y_val_pred = None, None

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    history['epoch_time_s'].append(elapsed)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    marker = ' [BEST]' if is_best else ''
    print(
        f"Epoch {epoch+1:3d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f} acc={val_acc:.3f} | "
        f"lr={current_lr:.2e} | {elapsed:.1f}s{marker}"
    )

    # Log a TensorBoard
    tb_logger.log_epoch(
        epoch=epoch,
        train_loss=train_loss,
        train_acc=train_acc,
        val_loss=val_loss,
        val_acc=val_acc,
        lr=current_lr,
    )

    # Matriz de confusion (keyword args — corrige el bug del V4)
    if y_val_true is not None:
        try:
            tb_logger.log_confusion_matrix(
                epoch,
                y_true=y_val_true,
                y_pred=y_val_pred,
                class_names=class_names,
            )
        except Exception as e:
            print(f"  (aviso: no se pudo loguear CM esta epoca: {e})")

    # Checkpoint a Drive cada epoca
    save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, epoch, history, best_val_acc, label_map)
    if is_best:
        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, epoch, history, best_val_acc, label_map)

    with open(HISTORY_JSON, 'w') as f:
        json.dump({
            'history': history,
            'best_val_acc': best_val_acc,
            'last_epoch_completed': epoch,
            'run_name': RUN_NAME,
            'label_map': label_map,
            'loss_info': {k: v for k, v in loss_info.items() if k != 'weights_vector'},
        }, f, indent=2)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\n[EARLY STOPPING] Val_acc no mejora en {EARLY_STOPPING_PATIENCE} epocas. Parando.")
        break

print("\n" + "=" * 75)
print(f"ENTRENAMIENTO TERMINADO. Mejor val_acc: {best_val_acc:.4f}")
print("=" * 75)

## Celda 12 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

COLOR_TRAIN = '#6C5CE7'
COLOR_VAL   = '#00CEC9'
COLOR_LR    = '#FD79A8'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#FAFAFA')
epochs_ran = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_ran, history['train_loss'], color=COLOR_TRAIN, lw=2, label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   color=COLOR_VAL,   lw=2, label='Val')
axes[0].set_title('Perdida (Attention v2 + CW)', fontsize=12)
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], color=COLOR_TRAIN, lw=2, label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   color=COLOR_VAL,   lw=2, label='Val')
axes[1].axhline(best_val_acc, color='#2D3436', ls='--', alpha=0.5,
                label=f'Best val_acc = {best_val_acc:.3f}')
axes[1].set_title('Accuracy (Attention v2 + CW)', fontsize=12)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history['lr'], color=COLOR_LR, lw=2)
axes[2].set_title('Learning Rate', fontsize=12)
axes[2].set_xlabel('Epoca'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
fig_path = f'{DRIVE_RUN_DIR}/curvas_entrenamiento.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f"OK Figura guardada en {fig_path}")
plt.show()

## Celda 13 — Evaluar en test set con el mejor modelo

In [ ]:
print("Cargando best.pth para evaluacion en test...")
best_ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f"Cargado: epoca {best_ckpt['epoch']+1}, val_acc={best_ckpt['best_val_acc']:.4f}")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nTest loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")

## Celda 14 — Classification report + Matriz de confusión finales

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().tolist())

print("=" * 70)
print("CLASSIFICATION REPORT (test set) — ATTENTION V2 (class weights + LS)")
print("=" * 70)
report = classification_report(
    all_labels, all_preds, target_names=class_names, digits=3, zero_division=0
)
print(report)

with open(f'{DRIVE_RUN_DIR}/classification_report.txt', 'w') as f:
    f.write(report)

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#FAFAFA')
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={'label': 'Conteo'}, ax=ax,
)
ax.set_xlabel('Prediccion'); ax.set_ylabel('Etiqueta real')
ax.set_title(f'Confusion Matrix — Attention v2 Test (acc={test_acc:.3f})')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
cm_path = f'{DRIVE_RUN_DIR}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f"\nOK Matriz de confusion guardada en {cm_path}")
plt.show()

## Celda 15 — Comparación automática: Baseline vs Attention v1 vs Attention v2

Intenta cargar los resultados del v1 desde Drive. Si no están disponibles,
muestra solo v2 y baseline.

In [ ]:
# Resultados del baseline (conocidos de ejecucion anterior)
BASELINE = {
    'model': 'SelvaSonicCNN (baseline)',
    'val_acc': 0.7419,
    'test_acc': 0.6322,
    'n_params': 422_635,
    'run_name': 'baseline_S3_v2_20260527_0118',
}

# Intentar cargar resultados del attention v1 desde Drive
V1_CKPT_DIR = f'{DRIVE_BASE}/checkpoints_activos_attention'
V1_HISTORY  = f'{V1_CKPT_DIR}/history.json'
v1_results  = None

try:
    if os.path.exists(V1_HISTORY):
        with open(V1_HISTORY) as f:
            v1_data = json.load(f)
        v1_val_acc = v1_data.get('best_val_acc')
        v1_test_acc = None

        # Buscar test_acc en el summary.json del run correspondiente
        runs_dir = f'{DRIVE_BASE}/runs'
        if os.path.exists(runs_dir):
            for run_dir_name in sorted(os.listdir(runs_dir), reverse=True):
                if 'attention_S4_v1' in run_dir_name:
                    summary_path = f'{runs_dir}/{run_dir_name}/summary.json'
                    if os.path.exists(summary_path):
                        with open(summary_path) as f:
                            s = json.load(f)
                        v1_test_acc = s.get('test_acc')
                        break

        v1_results = {
            'model': 'SelvaSonicCNNAttention v1 (sin CW)',
            'val_acc': v1_val_acc,
            'test_acc': v1_test_acc,
            'n_params': None,
        }
        print(f"OK Resultados v1 cargados (best_val_acc={v1_val_acc:.4f})")
    else:
        print(f"INFO: No hay history.json del v1 en {V1_HISTORY}")
        print("     Solo se muestra comparacion con baseline.")
except Exception as e:
    print(f"AVISO: No se pudieron cargar resultados del v1: {e}")

# Guardar resumen final del v2
summary = {
    'run_name': RUN_NAME,
    'timestamp': datetime.now().isoformat(),
    'epochs_completed': len(history['train_loss']),
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'n_params': n_params,
    'use_class_weights': USE_CLASS_WEIGHTS,
    'label_smoothing': LABEL_SMOOTHING,
    'hparams': hparams_dict,
    'loss_info': {k: v for k, v in loss_info.items() if k != 'weights_vector'},
    'comparison_baseline': BASELINE,
}
with open(f'{DRIVE_RUN_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

shutil.copy(BEST_CKPT,   f'{DRIVE_RUN_DIR}/best.pth')
shutil.copy(LATEST_CKPT, f'{DRIVE_RUN_DIR}/latest.pth')
shutil.copy(HISTORY_JSON, f'{DRIVE_RUN_DIR}/history.json')

tb_logger.log_hparams_final({'best_val_acc': best_val_acc, 'test_acc': test_acc})
tb_logger.close()

# Tabla de comparacion
models_to_compare = [BASELINE]
if v1_results:
    models_to_compare.append(v1_results)
models_to_compare.append({
    'model': 'SelvaSonicCNNAttention v2 (class weights + LS)',
    'val_acc': best_val_acc,
    'test_acc': test_acc,
    'n_params': n_params,
})

print()
print("=" * 75)
print("COMPARACION FINAL DE MODELOS")
print("=" * 75)
print(f"  {'Modelo':<42} | {'val_acc':>8} | {'test_acc':>9} | {'params':>10}")
print("  " + "-" * 75)
for m in models_to_compare:
    val  = f"{m['val_acc']:.4f}"  if m.get('val_acc')  is not None else "   N/A"
    tst  = f"{m['test_acc']:.4f}" if m.get('test_acc') is not None else "    N/A"
    par  = f"{m['n_params']:,}"   if m.get('n_params')  is not None else "N/A"
    print(f"  {m['model']:<42} | {val:>8} | {tst:>9} | {par:>10}")
print("=" * 75)

delta_val  = best_val_acc - BASELINE['val_acc']
delta_test = test_acc - BASELINE['test_acc']
print(f"\nDelta v2 vs baseline: val_acc {delta_val:+.4f} | test_acc {delta_test:+.4f}")
if v1_results and v1_results.get('val_acc'):
    dv1v2 = best_val_acc - v1_results['val_acc']
    print(f"Delta v2 vs v1:       val_acc {dv1v2:+.4f}")

print(f"\nOK Run completo guardado en {DRIVE_RUN_DIR}")